# Speech Recognition — Colab Training

Train the LSTM-CTC model on **Google Colab GPU**. Your computer can be off — training runs on Google's servers.

**Before running:** Runtime → Change runtime type → **T4 GPU**

**Steps:**
1. Clone repo
2. Install dependencies
3. Download LibriSpeech dataset (~6 GB, one-time per session)
4. Train on full dataset
5. Save model to Google Drive (optional but recommended)

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Enable Runtime → Change runtime type → T4 GPU')

In [ ]:
# Clone the project (replace with your fork URL if needed)
!git clone https://github.com/sumathis15/Speech_Recognition.git
%cd Speech_Recognition

In [ ]:
!pip install -q -r requirements.txt

## Download LibriSpeech (train-clean-100)
Downloads ~6 GB from OpenSLR. Takes 10–20 minutes depending on connection.

In [ ]:
import os

DATA_DIR = 'data/raw/LibriSpeech/train-clean-100'
MARKER = os.path.join(DATA_DIR, '.download_complete')

if os.path.exists(MARKER):
    print('Dataset already downloaded in this session.')
else:
    os.makedirs('data/raw', exist_ok=True)
    print('Downloading train-clean-100 (~6 GB)...')
    !wget -q --show-progress -O data/raw/train-clean-100.tar.gz \
        https://www.openslr.org/resources/11/train-clean-100.tar.gz
    print('Extracting...')
    !tar -xf data/raw/train-clean-100.tar.gz -C data/raw/
    !rm data/raw/train-clean-100.tar.gz
    open(MARKER, 'w').close()
    print('Done.')

flac_count = sum(1 for r, _, files in os.walk(DATA_DIR) for f in files if f.endswith('.flac'))
print(f'FLAC files found: {flac_count}')

## (Optional) Mount Google Drive to save the model
Recommended — Colab sessions disconnect and you lose files otherwise.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_MODEL_DIR = '/content/drive/MyDrive/Speech_Recognition/model'
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
print('Model will be copied to:', DRIVE_MODEL_DIR)

## Train on full dataset
GPU: use batch size 32. Expect ~15–30 min per epoch.

In [ ]:
!python train.py --epochs 30 --batch-size 32 --num-workers 2

## Save model to Google Drive + download locally

In [ ]:
import shutil
from google.colab import files

if os.path.exists('model/lstm_ctc_model.pth'):
    if 'DRIVE_MODEL_DIR' in dir():
        shutil.copy('model/lstm_ctc_model.pth', f'{DRIVE_MODEL_DIR}/lstm_ctc_model.pth')
        shutil.copy('model/lstm_ctc_model_best.pth', f'{DRIVE_MODEL_DIR}/lstm_ctc_model_best.pth')
        print('Saved to Google Drive.')
    files.download('model/lstm_ctc_model.pth')
    print('Download started — place file in your local model/ folder for app.py')
else:
    print('Model not found. Training may not have finished.')